## Setup

In [1]:
import os
import math
import pandas as pd
from pathlib import Path
from csv import QUOTE_NONE

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import DoubleType


# Load

In [2]:
LOCAL_DATA_ROOT = Path(os.environ.get("MIND_DATA_ROOT", "../smallDataset")).expanduser().resolve()

train_dir = LOCAL_DATA_ROOT / f"MINDsmall_train"
valid_dir = LOCAL_DATA_ROOT / f"MINDsmall_dev"

train_news_path      = str(train_dir / "news.tsv")
train_behaviors_path = str(train_dir / "behaviors.tsv")
valid_news_path      = str(valid_dir / "news.tsv")
valid_behaviors_path = str(valid_dir / "behaviors.tsv")

NEWS_COLUMNS = [
    "nid",
    "vertical",
    "subvertical",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities",
]
BEHAVIOR_COLUMNS = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

def load_tsv(path_str: str, columns):
    return pd.read_table(
        path_str,
        header=None,
        names=columns,
        sep='\t',
        quoting=QUOTE_NONE,
        dtype=object,
        na_filter=False,
    )

train_news_df = load_tsv(train_news_path, NEWS_COLUMNS)
valid_news_df = load_tsv(valid_news_path, NEWS_COLUMNS)
train_behaviors_df = load_tsv(train_behaviors_path, BEHAVIOR_COLUMNS)
valid_behaviors_df = load_tsv(valid_behaviors_path, BEHAVIOR_COLUMNS)

print(f"Loaded {len(train_news_df)} train news rows and {len(valid_news_df)} validation news rows from {LOCAL_DATA_ROOT}")
print(f"Loaded {len(train_behaviors_df)} train behaviors rows and {len(valid_behaviors_df)} validation behaviors rows")

Loaded 51282 train news rows and 42416 validation news rows from /home/asmvas/Documents/NTNU-prosjekt/anbefalingssytem/AnbSysProsjekt/smallDataset
Loaded 156965 train behaviors rows and 73152 validation behaviors rows


## Build interactions for ALS

In [3]:
def build_interactions(df: pd.DataFrame) -> pd.DataFrame:
    """Expands impression strings into (user, news, label) rows."""
    rows = []
    for row in df.itertuples(index=False):
        impressions = (row.impressions or '').strip()
        if not impressions:
            continue
        for token in impressions.split():
            if '-' not in token:
                continue
            news_id, label = token.rsplit('-', 1)
            if not news_id:
                continue
            try:
                label_value = float(label)
            except ValueError:
                continue
            rows.append((row.user_id, news_id, label_value))
    return pd.DataFrame(rows, columns=['user_id', 'nid', 'label'])


train_interactions_pd = build_interactions(train_behaviors_df)
valid_interactions_pd = build_interactions(valid_behaviors_df)
print(f"Prepared {len(train_interactions_pd):,} train interactions and {len(valid_interactions_pd):,} validation interactions")

Prepared 5,843,444 train interactions and 2,740,998 validation interactions


## Create Spark session

In [4]:
spark = (
    SparkSession.builder
    .appName('mind-als-demo')
    .getOrCreate()
)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/06 14:16:06 WARN Utils: Your hostname, asmvas-VivoBook-ASUSLaptop-X421EA-S433EA, resolves to a loopback address: 127.0.1.1; using 10.22.108.140 instead (on interface wlo1)
26/03/06 14:16:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 14:16:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Prepare Spark DataFrames

In [5]:
train_spark_df = spark.createDataFrame(train_interactions_pd)
valid_spark_df = spark.createDataFrame(valid_interactions_pd)

user_indexer = StringIndexer(
    inputCol='user_id',
    outputCol='user_idx',
    handleInvalid='skip'
).fit(train_spark_df)
item_indexer = StringIndexer(
    inputCol='nid',
    outputCol='item_idx',
    handleInvalid='skip'
).fit(train_spark_df)

train_indexed_df = item_indexer.transform(user_indexer.transform(train_spark_df))
valid_indexed_df = item_indexer.transform(user_indexer.transform(valid_spark_df))

train_ratings_df = train_indexed_df.select(
    F.col('user_idx').cast('int').alias('user_idx'),
    F.col('item_idx').cast('int').alias('item_idx'),
    F.col('label').cast('float').alias('label'),
).cache()

valid_ratings_df = valid_indexed_df.select(
    F.col('user_idx').cast('int').alias('user_idx'),
    F.col('item_idx').cast('int').alias('item_idx'),
    F.col('label').cast('float').alias('label'),
).cache()

print(f"Spark train interactions: {train_ratings_df.count():,}")
print(f"Spark validation interactions: {valid_ratings_df.count():,}")

26/03/06 14:18:18 WARN TaskSetManager: Stage 0 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/06 14:18:25 WARN TaskSetManager: Stage 6 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/06 14:18:30 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/06 14:18:30 WARN TaskSetManager: Stage 12 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/06 14:18:36 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/06 14:18:36 WARN TaskSetManager: Stage 13 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/06 14:18:36 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/06 14:18:36 WARN TaskSetManager: Stage 16 contains a task of very large size (8184 KiB). The maximum recommended task size is 1000 KiB.


Spark train interactions: 5,843,444


26/03/06 14:18:39 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/06 14:18:39 WARN TaskSetManager: Stage 17 contains a task of very large size (8184 KiB). The maximum recommended task size is 1000 KiB.


Spark validation interactions: 265,183


## Train and evaluate ALS

In [6]:
als = ALS(
    rank=32,
    maxIter=10,
    regParam=0.05,
    userCol='user_idx',
    itemCol='item_idx',
    ratingCol='label',
    implicitPrefs=False,
    nonnegative=True,
    coldStartStrategy='drop',
)

als_model = als.fit(train_ratings_df)

predictions = als_model.transform(valid_ratings_df)

evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='label',
    predictionCol='prediction',
)
rmse = evaluator.evaluate(predictions)
print(f"Validation RMSE: {rmse:.4f}")

26/03/06 14:18:39 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:18:39 WARN TaskSetManager: Stage 20 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/06 14:18:40 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:18:40 WARN TaskSetManager: Stage 21 contains a task of very large size (17481 KiB). The maximum recommended task size is 1000 KiB.
26/03/06 14:18:41 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:18:43 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:18:45 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:18:46 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:18:48 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:18:49 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:18:50 WARN DAG

Validation RMSE: 0.1949


## Ranking metrics

In [7]:
LOG_2 = math.log(2.0)
rank_window = Window.partitionBy('user_idx').orderBy(F.desc('prediction'))
ranked_predictions_df = predictions.withColumn('rank', F.row_number().over(rank_window)).cache()

user_label_stats_df = valid_ratings_df.groupBy('user_idx').agg(
    F.sum('label').alias('num_pos'),
    F.count('*').alias('num_interactions'),
)


def make_ideal_dcg_udf(k: int):
    def ideal(count):
        if count is None:
            return 0.0
        upto = min(int(count), k)
        score = 0.0
        for i in range(upto):
            score += 1.0 / math.log2(i + 2)
        return float(score)

    return F.udf(ideal, DoubleType())


ideal_dcg_5_udf = make_ideal_dcg_udf(5)
ideal_dcg_10_udf = make_ideal_dcg_udf(10)

user_label_stats_df = (
    user_label_stats_df
    .withColumn('ideal_dcg_5', ideal_dcg_5_udf('num_pos'))
    .withColumn('ideal_dcg_10', ideal_dcg_10_udf('num_pos'))
)


def summarize_topk(topk_df, k: int):
    """Aggregate DCG and hit contributions for cutoff k."""
    contrib_df = topk_df.withColumn(
        'dcg_contrib',
        F.col('label') * (F.lit(LOG_2) / F.log(F.col('rank') + F.lit(1.0))),
    )
    return contrib_df.groupBy('user_idx').agg(
        F.sum('dcg_contrib').alias(f'dcg_{k}'),
        F.max('label').alias(f'hit_{k}'),
    )


top5_metrics_df = summarize_topk(ranked_predictions_df.filter(F.col('rank') <= 5), 5)
top10_metrics_df = summarize_topk(ranked_predictions_df.filter(F.col('rank') <= 10), 10)

metrics_per_user_df = (
    user_label_stats_df
    .join(top5_metrics_df, 'user_idx', 'left')
    .join(top10_metrics_df, 'user_idx', 'left')
    .fillna({'dcg_5': 0.0, 'dcg_10': 0.0, 'hit_5': 0.0, 'hit_10': 0.0})
)

metrics_per_user_df = (
    metrics_per_user_df
    .withColumn('ndcg_5', F.when(F.col('ideal_dcg_5') > 0, F.col('dcg_5') / F.col('ideal_dcg_5')))
    .withColumn('ndcg_10', F.when(F.col('ideal_dcg_10') > 0, F.col('dcg_10') / F.col('ideal_dcg_10')))
)

ranking_summary_row = metrics_per_user_df.agg(
    F.mean('ndcg_5').alias('ndcg_5'),
    F.mean('ndcg_10').alias('ndcg_10'),
    F.mean('hit_5').alias('hit_5'),
    F.mean('hit_10').alias('hit_10'),
).first()
ranking_summary = ranking_summary_row.asDict() if ranking_summary_row else {}

first_relevant_df = (
    ranked_predictions_df
    .filter(F.col('label') > 0)
    .groupBy('user_idx')
    .agg(F.min('rank').alias('first_rank'))
)

rr_df = metrics_per_user_df.select('user_idx').join(first_relevant_df, 'user_idx', 'left').withColumn(
    'rr',
    F.when(F.col('first_rank').isNotNull(), 1.0 / F.col('first_rank')).otherwise(0.0),
)
mrr_row = rr_df.agg(F.mean('rr').alias('mrr')).first()
mrr_value = mrr_row['mrr'] if mrr_row else None

score_and_labels_rdd = predictions.select('prediction', 'label').rdd.map(
    lambda row: (float(row.prediction), float(row.label))
)
auc_value = BinaryClassificationMetrics(score_and_labels_rdd).areaUnderROC


def to_float(value):
    return 0.0 if value is None else float(value)

ndcg5 = to_float(ranking_summary.get('ndcg_5'))
ndcg10 = to_float(ranking_summary.get('ndcg_10'))
hit5 = to_float(ranking_summary.get('hit_5'))
hit10 = to_float(ranking_summary.get('hit_10'))
mrr = to_float(mrr_value)
auc = to_float(auc_value)

print(f"nDCG@5: {ndcg5:.4f}")
print(f"nDCG@10: {ndcg10:.4f}")
print(f"Hit@5: {hit5:.4f}")
print(f"Hit@10: {hit10:.4f}")
print(f"MRR: {mrr:.4f}")
print(f"AUC: {auc:.4f}")

ranked_predictions_df.unpersist()


26/03/06 14:19:23 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:19:23 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:19:23 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/06 14:19:24 WARN TaskSetManager: Stage 132 contains a task of very large size (8184 KiB). The maximum recommended task size is 1000 KiB.
26/03/06 14:19:25 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/03/06 14:19:25 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/03/06 14:19:25 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/06 14:19:33 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/03/06 14:19:36 WARN TaskSetManager: Stage 163 contains a task of very large size (8184 KiB). The maximum recommended task size is 1000 KiB.
26/03/06 14:19:37 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/03/06 14:19:48 WARN DAG

nDCG@5: 0.2799
nDCG@10: 0.3361
Hit@5: 0.4420
Hit@10: 0.5959
MRR: 0.2764
AUC: 0.6004


DataFrame[user_idx: int, item_idx: int, label: float, prediction: float, rank: int]

## Inspect sample recommendations

In [8]:
als_model.recommendForAllUsers(5).limit(5).show(truncate=False)

26/03/06 14:19:59 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/03/06 14:20:12 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB


+--------+-------------------------------------------------------------------------------------------------------------+
|user_idx|recommendations                                                                                              |
+--------+-------------------------------------------------------------------------------------------------------------+
|26      |[{18298, 0.25378245}, {18910, 0.25057727}, {18785, 0.25038147}, {17833, 0.25038147}, {17456, 0.25038147}]    |
|27      |[{18298, 0.11796741}, {19246, 0.11770375}, {19827, 0.116469875}, {18910, 0.11574878}, {20267, 0.11567059}]   |
|28      |[{18298, 0.07660733}, {18451, 0.07548769}, {18910, 0.07539985}, {18254, 0.07535842}, {19635, 0.07505131}]    |
|31      |[{18298, 0.29242912}, {19827, 0.2899141}, {16441, 0.2889486}, {18910, 0.28876406}, {19246, 0.28711313}]      |
|34      |[{18298, 0.12122171}, {19246, 0.119864225}, {18947, 0.119420126}, {18785, 0.119154096}, {17833, 0.119154096}]|
+--------+----------------------